### (optional) rebuild the algorithm module first

`logistic_regression.py` — the module the import cell below pulls `LogisticRegressionClass`
from — is **auto-generated** from `../logistic_regression_hold_out.ipynb`. The next cell
regenerates it so the class matches the latest notebook edits, replacing the old manual
"remember to run the build script" step.

- **Save the source notebook first** — the build reads it from disk, not from your open editor buffer.
- Freshness applies on a **fresh kernel / first import**. If the module was already imported
  this session, **restart the kernel** after rebuilding (Python won't re-import a loaded module).
- Running it is **optional** — skip it to use the current `logistic_regression.py` as-is.

In [ ]:
# TODO: check how projection has been done. 
# I want to see how it is implemented. 

In [1]:
# --- (optional) rebuild the algorithm module from the source notebook ---
# logistic_regression.py is generated from ../logistic_regression_hold_out.ipynb.
# Run this cell so the imported LogisticRegressionClass reflects the latest notebook edits.
# NOTE: reads the notebook AS SAVED ON DISK -> save it first; and freshness only applies on
# a fresh kernel / first import (restart the kernel if the module was already loaded).
import subprocess, sys
subprocess.run([sys.executable, "build_logistic_module.py"], check=True)
print("rebuilt logistic_regression.py from the source notebook")

wrote /home/alimorty/early_stopping/ali_code/logistic_regression.py
rebuilt logistic_regression.py from the source notebook


In [2]:
import os, sys, json, hashlib, pickle
# logistic_regression.py (Ali's nbconvert'd algorithm) lives in ali_code/, the parent of this LLM_visualization/ folder
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# pin a single renderer so fig.show()/display() doesn't double-emit the figure in Jupyter/VSCode
pio.renderers.default = "notebook"

from logistic_regression import LogisticRegressionClass

# Region-of-trajectories, with *time* along the line

Each GD trajectory is projected onto the 2D plane
(axis 1 = `w*`, axis 2 = `sum_i w_tilde_i`) by `LLM_generated_region_experiment_v1`.

A plain line hides **how many steps** were spent in each part of the path. For logistic
GD on separable data `||w_t|| ~ log(t)`, so equal-size spatial moves take
exponentially more steps as `t` grows: ~90% of a 100k-step run is crammed into a tiny
end-segment. This notebook makes that visible:

- **equal-step tick markers** — dense clusters = many steps per unit length = lots of time spent there.
- **color = log10(step)** — dark (early) → bright (late).
- **hover** — point at any marker to read the exact step number. Zoom into the crammed
  end-blob and hover to drill in.


In [3]:
# --- experiment setup (edit freely) ---
# Only scalars here (cheap). The heavy GD runs happen in the plot cells below and are
# cached, so re-running an identical setup reloads instead of recomputing.
n = 100
d = 200
k = 10
number_of_trajectories = 3
t_steps = int(1e5)      # the whole point: many steps, so the log-time bunching shows
eta = 0.1
seed = 85               # class random_seed; part of a run's identity


In [4]:
TRAJ_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e",
               "#8c564b", "#e377c2", "#17becf"]


def plot_trajectories_with_time(projected_trajectories, n_ticks=400, tick_every=None,
                                mark_points=None, w_tilde_dirs=None,
                                early_stop_pts=None, argmin_pts=None,
                                early_stop_traj=None, argmin_traj=None, title=None,
                                x_label="w_1", y_label="w_2", basis_arrows=None,
                                width=820, height=820):
    """Interactive plotly view of projected GD trajectories with time encoded.

    Each trajectory gets one color (matched to its w_tilde ray), so trajectory i
    is visually tied to its own asymptote w~_i.

    - full path (Scattergl so 100k+ points stay responsive)
    - equal-step tick markers; their DENSITY (not color) shows dwell time
    - hover on any tick -> exact step number

    n_ticks     : approx number of equal-step ticks per trajectory (ignored if tick_every set)
    tick_every  : put a tick every this many steps (overrides n_ticks)
    mark_points : optional dict {label: (x, y)} of extra reference points in the SAME
                  projected plane (e.g. {"w*": (||w*||, w*.w_2_dir)}), drawn as gold
                  diamonds so their position/magnitude can be compared to the trajectories.
    w_tilde_dirs: optional list of projected (x, y) for each trajectory's w_tilde
                  (max-margin direction). Drawn as a dashed ray from the origin through
                  that direction (color-matched per trajectory) plus a marker at the
                  actual w_tilde point, so you can see whether trajectory i is bending
                  toward its own w_tilde_i line.
    early_stop_pts: optional list of projected (x, y) of each trajectory's oracular
                  early-stop iterate (green square). Drawn in that trajectory's legend group, so it hides with the trajectory.
    argmin_pts  : optional list of projected (x, y) of each trajectory's population
                  test-loss argmin iterate (purple star). Drawn in that trajectory's legend group, so it hides with the trajectory.
    """
    fig = go.Figure()
    # how far to extend the w_tilde rays: a bit past the farthest trajectory point
    ray_len = 1.15 * max(float(np.max(np.linalg.norm(proj, axis=1)))
                         for proj in projected_trajectories)

    for idx, proj in enumerate(projected_trajectories):
        color = TRAJ_COLORS[idx % len(TRAJ_COLORS)]
        T = proj.shape[0]
        stride = tick_every if tick_every is not None else max(1, T // n_ticks)
        tick_idx = np.arange(0, T, stride)

        # full path, in this trajectory's color
        fig.add_trace(go.Scattergl(
            x=proj[:, 0], y=proj[:, 1], mode="lines",
            line=dict(width=1, color=color),
            name=f"traj {idx} path", legendgroup=f"traj{idx}",
            showlegend=False, hoverinfo="skip",
        ))
        # equal-step ticks: same color; DENSITY conveys time (dense = many steps here)
        fig.add_trace(go.Scattergl(
            x=proj[tick_idx, 0], y=proj[tick_idx, 1], mode="markers",
            marker=dict(size=5, color=color),
            text=[f"traj {idx}<br>step {int(t):,}" for t in tick_idx],
            hoverinfo="text", name=f"traj {idx}", legendgroup=f"traj{idx}",
        ))

    # optional w_tilde_i directions: dashed ray from origin + marker, per trajectory
    if w_tilde_dirs is not None:
        for idx, (px, py) in enumerate(w_tilde_dirs):
            color = TRAJ_COLORS[idx % len(TRAJ_COLORS)]
            norm = float(np.hypot(px, py))
            if norm == 0:
                continue
            ux, uy = px / norm, py / norm
            # asymptote line (the direction the trajectory should converge to)
            fig.add_trace(go.Scattergl(
                x=[0.0, ray_len * ux], y=[0.0, ray_len * uy], mode="lines",
                line=dict(width=1.5, color=color, dash="dash"),
                name=f"w~_{idx} dir", legendgroup=f"wtilde{idx}",
                hoverinfo="skip",
            ))
            # marker at the actual w_tilde_i point
            fig.add_trace(go.Scattergl(
                x=[px], y=[py], mode="markers",
                marker=dict(size=9, color=color, symbol="x",
                            line=dict(width=1, color="black")),
                text=[f"w~_{idx}<br>({px:.3g}, {py:.3g})"], hoverinfo="text",
                name=f"w~_{idx}", legendgroup=f"wtilde{idx}", showlegend=False,
            ))

    # optional stopping points: one marker trace PER trajectory, placed in that
    # trajectory's legendgroup so hiding a trajectory also hides its early-stop /
    # argmin markers (relies on legend groupclick="togglegroup", set in the layout).
    def _add_stop_markers(pts, trajs, symbol, name, size, per_traj_color, fixed_color):
        trajs = trajs if trajs is not None else list(range(len(pts)))
        for (px, py), i in zip(pts, trajs):
            color = TRAJ_COLORS[i % len(TRAJ_COLORS)] if per_traj_color else fixed_color
            fig.add_trace(go.Scattergl(
                x=[px], y=[py], mode="markers",
                marker=dict(size=size, color=color, symbol=symbol,
                            line=dict(width=1.5, color="black")),
                text=[f"{name} (traj {i})"], hoverinfo="text",
                name=name, legendgroup=f"traj{i}", showlegend=False,
            ))
    if early_stop_pts:
        _add_stop_markers(early_stop_pts, early_stop_traj, "square",
                          "early stop", 12, per_traj_color=True, fixed_color="green")
    if argmin_pts:
        _add_stop_markers(argmin_pts, argmin_traj, "star",
                          "test-loss argmin", 15, per_traj_color=True, fixed_color="purple")

    # legend key (no data points): explains that square = early stop, star = argmin.
    # Neutral gray so it reads as a symbol legend, not tied to any trajectory color.
    # These are legend-only swatches; clicking them toggles nothing (they carry no data).
    if early_stop_pts:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=12, color="#555", symbol="square",
                        line=dict(width=1.5, color="black")),
            name="early stop", hoverinfo="skip",
        ))
    if argmin_pts:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=14, color="#555", symbol="star",
                        line=dict(width=1.5, color="black")),
            name="test-loss argmin", hoverinfo="skip",
        ))

    # optional reference points (e.g. w*) in the same projected plane
    if mark_points:
        for label, (px, py) in mark_points.items():
            fig.add_trace(go.Scattergl(
                x=[px], y=[py], mode="markers+text",
                marker=dict(size=14, color="gold", symbol="diamond",
                            line=dict(width=1.5, color="black")),
                text=[label], textposition="top center",
                hovertext=[f"{label}<br>({px:.3g}, {py:.3g})"], hoverinfo="text",
                name=label, showlegend=True,
            ))

    # optional basis arrows from the origin: v1 (the anchor, on +x) and v2 (the raw
    # y-pick, tilted by its correlation with v1). The label sits at the arrowHEAD.
    if basis_arrows:
        alen = 0.6 * ray_len
        for key, color in (("v1", "#444"), ("v2", "#888")):
            if key not in basis_arrows:
                continue
            ux, uy, lbl = basis_arrows[key]
            hx, hy = alen * ux, alen * uy
            fig.add_annotation(x=hx, y=hy, ax=0.0, ay=0.0,
                               xref="x", yref="y", axref="x", ayref="y",
                               showarrow=True, arrowhead=2, arrowwidth=2, arrowcolor=color)
            fig.add_annotation(x=hx, y=hy, xref="x", yref="y", showarrow=False,
                               text=f"{key} = {lbl}", font=dict(color=color, size=13),
                               xshift=10 * ux, yshift=10 * uy,
                               xanchor=("left" if ux >= 0 else "right"))

    fig.update_layout(
        title=title or "GD trajectories in (w*, sum w_tilde) plane — tick density & color = time",
        xaxis_title=x_label, yaxis_title=y_label,
        width=width, height=height, template="plotly_white",
        legend=dict(title="trajectory", groupclick="togglegroup"),
    )
    return fig


In [5]:
# --- run cache: store the compact data needed to redraw ANY axis pair ---
# GD is the slow part; once a setup is run we pickle its compact plot data so an identical
# setup reloads instantly. index.json lets the saved-runs dropdown browse past runs.
#   v3 schema: P (per-axis projections of every iterate), G (axis Gram matrix), axis_labels,
#              ref_rows (w*/w~_i raw projections at TRUE magnitude), stop indices. The chosen
#              (x, y) axis pair is reconstructed at VIEW time -> browse projections for free.
#   v1/v2 schema (legacy): a single fixed 2D projection (projected_trajectories + w*/w~ dirs).
CACHE_DIR = "region_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
INDEX_FILE = os.path.join(CACHE_DIR, "index.json")

# fields stored per run (also the searchable label). n_random_axes only matters for v3.
SETUP_KEYS = ["version", "n", "d", "k", "number_of_trajectories", "t_steps",
              "eta", "seed", "test_sample_size", "normalize_w_tilde", "use_lambda_diag",
              "n_random_axes", "basis_axes"]
# a run's IDENTITY ignores t_steps: asking for more steps extends the same entry, so
# there is exactly one cache entry per setting and it holds the highest T computed.
IDENTITY_KEYS = [key for key in SETUP_KEYS if key != "t_steps"]

def config_key(setup):
    payload = {k: setup[k] for k in IDENTITY_KEYS}
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:8]

def load_index():
    try:
        with open(INDEX_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def _update_index(key, setup):
    idx = load_index()
    idx[key] = {k: setup[k] for k in SETUP_KEYS}
    with open(INDEX_FILE, "w") as f:
        json.dump(idx, f, indent=2)

def run_label(key, s):
    return (f"[{key}] {s['version']} | n{s['n']} d{s['d']} k{s['k']} "
            f"traj{s['number_of_trajectories']} T{s['t_steps']} eta{s['eta']} "
            f"seed{s['seed']} test{s['test_sample_size']}")

def reconstruct_2d(P_rows, G, a, b):
    """View-time Gram-Schmidt: turn raw per-axis projections into orthonormal 2D coords.

    P_rows : (..., n_axes) array of <vector, v_i> against the (unit) stored axes.
    a, b   : axis indices to use as x and y. Returns (..., 2).
    Orthonormalizes the (v_a, v_b) plane from the axis Gram matrix G, so v_a is the x-axis
    and v_b's component orthogonal to v_a is the y-axis (exactly proj_on_2d_subspace)."""
    P_rows = np.asarray(P_rows, dtype=float)
    Gaa, Gbb, Gab = G[a, a], G[b, b], G[a, b]
    x = P_rows[..., a] / np.sqrt(Gaa)
    denom = np.sqrt(max(Gbb - Gab ** 2 / Gaa, 0.0))
    if denom < 1e-12:            # a == b (or identical axes): the y-axis collapses
        y = np.zeros_like(x)
    else:
        y = (P_rows[..., b] - (Gab / Gaa) * P_rows[..., a]) / denom
    return np.stack([x, y], axis=-1)

def run_or_load_region(setup, force=False):
    """Load compact plot data for `setup` from cache, or run the experiment + save it.
    setup['version'] selects v1/v2 (fixed plane) or v3 (browsable axes).
    Returns (plotdata, key). Pass force=True to recompute and overwrite."""
    setup = dict(setup)
    key = config_key(setup)
    path = os.path.join(CACHE_DIR, f"region_{key}.pkl")
    if os.path.exists(path) and not force:
        with open(path, "rb") as f:
            pd = pickle.load(f)
        # cached entry already has >= the requested steps: reuse it (it's the highest T).
        if pd["setup"]["t_steps"] >= setup["t_steps"]:
            print("loaded", run_label(key, pd["setup"]))
            return pd, key
        # asking for more steps: recompute at the larger T and overwrite the one entry.
        # (TODO future: resume GD from the stored last iterate instead of recomputing.)
        print(f"extending T {pd['setup']['t_steps']} -> {setup['t_steps']} (recompute)")

    w_star = np.zeros(setup["d"]); w_star[:setup["k"]] = 1.0
    lam = np.arange(1, setup["d"] + 1, dtype=float) ** (-2)
    inst = LogisticRegressionClass(setup["n"], setup["d"],
                                   random_seed=setup["seed"], eta=setup["eta"])
    common = dict(number_of_trajectories=setup["number_of_trajectories"], w_star=w_star,
                  d=setup["d"], n=setup["n"], t_steps=setup["t_steps"], eta=setup["eta"],
                  use_lambda_diag=setup["use_lambda_diag"], lambda_diag=lam,
                  normalize_w_tilde=setup["normalize_w_tilde"], plot=False)

    if setup["version"] == "v3":
        result = inst.LLM_generated_region_experiment_v3(
            test_sample_size=setup["test_sample_size"],
            measure_population_loss_for_iterates=True,
            n_random_axes=setup["n_random_axes"],
            basis_axes=setup["basis_axes"], **common)
        P = np.asarray(result["P"])                 # (n_traj, T, n_axes)
        G = np.asarray(result["G"])                 # (n_axes, n_axes)
        axis_labels = list(result["axis_labels"])
        w_star_vec = np.asarray(result["w_star"])
        w_tilde_vecs = [np.asarray(w) for w in result["w_tildes"]]
        # reference markers at TRUE magnitude: each ref is one of the unit axes, so its raw
        # projection onto axis v_a is ||ref|| * G[ref_axis_idx, a] (recovers ||ref|| lost to
        # axis normalization). Stored as full rows so any axis pair reconstructs correctly.
        ref_rows = {"w*": float(np.linalg.norm(w_star_vec)) * G[axis_labels.index("w*")]}
        for i in range(setup["number_of_trajectories"]):
            lbl = f"w~_{i}"
            ref_rows[lbl] = float(np.linalg.norm(w_tilde_vecs[i])) * G[axis_labels.index(lbl)]
        pd = {
            "setup": {k: setup[k] for k in SETUP_KEYS},
            "P": P, "G": G, "axis_labels": axis_labels,
            "ref_rows": {k: np.asarray(v) for k, v in ref_rows.items()},
            "early_stop_t": list(result["early_stop_t"]),
            "test_loss_argmin_t": list(result["test_loss_argmin_t"]),
        }
        with open(path, "wb") as f:
            pickle.dump(pd, f)
        _update_index(key, setup)
        print("computed + saved", run_label(key, pd["setup"]))
        return pd, key

    # ---- legacy v1 / v2: single fixed projection plane ----
    if setup["version"] == "v2":
        result = inst.LLM_generated_region_experiment_v2(
            test_sample_size=setup["test_sample_size"],
            measure_population_loss_for_iterates=True, **common)
        early_stop_t = list(result["early_stop_t"])
        test_loss_argmin_t = list(result["test_loss_argmin_t"])
    else:
        result = inst.LLM_generated_region_experiment_v1(**common)
        early_stop_t = [None] * setup["number_of_trajectories"]
        test_loss_argmin_t = [None] * setup["number_of_trajectories"]

    w1, w2 = result["w_1_direction"], result["w_2_direction"]
    w_star_proj = inst.proj_on_2d_subspace(w_star, w1, w2)          # (2,)
    w_tilde_proj = inst.proj_on_2d_subspace(
        np.asarray(result["w_tildes"]), w1, w2)                     # (n_traj, 2)
    pd = {
        "setup": {k: setup[k] for k in SETUP_KEYS},
        "projected_trajectories": [np.asarray(p) for p in result["projected_trajectories"]],
        "w_star_proj": (float(w_star_proj[0]), float(w_star_proj[1])),
        "w_tilde_dirs": [(float(x), float(y)) for x, y in w_tilde_proj],
        "early_stop_t": early_stop_t,
        "test_loss_argmin_t": test_loss_argmin_t,
    }
    with open(path, "wb") as f:
        pickle.dump(pd, f)
    _update_index(key, setup)
    print("computed + saved", run_label(key, pd["setup"]))
    return pd, key

def load_region_run(key):
    with open(os.path.join(CACHE_DIR, f"region_{key}.pkl"), "rb") as f:
        return pickle.load(f)

def plot_from_plotdata(pd, n_ticks=400, title=None, x_label=None, y_label=None):
    """Rebuild the interactive figure from cached compact plot data.
    For v3 (P/G present) the (x_label, y_label) axis pair is reconstructed at view time;
    for legacy v1/v2 the stored fixed projection is used and the labels are ignored."""
    if "P" in pd:                                  # v3: reconstruct chosen axis pair
        labels, G, P = pd["axis_labels"], pd["G"], pd["P"]
        x_label = x_label if x_label in labels else ("w*" if "w*" in labels else labels[0])
        y_label = y_label if y_label in labels else ("sum_w~" if "sum_w~" in labels else labels[-1])
        a, b = labels.index(x_label), labels.index(y_label)
        proj = [reconstruct_2d(P[i], G, a, b) for i in range(P.shape[0])]
        w_star_2d = tuple(reconstruct_2d(pd["ref_rows"]["w*"], G, a, b))
        w_tilde_dirs = [tuple(reconstruct_2d(pd["ref_rows"][f"w~_{i}"], G, a, b))
                        for i in range(P.shape[0])]
        es_i = [i for i, t in enumerate(pd["early_stop_t"]) if t is not None]
        am_i = [i for i, t in enumerate(pd["test_loss_argmin_t"]) if t is not None]
        es = [tuple(proj[i][pd["early_stop_t"][i]]) for i in es_i]
        am = [tuple(proj[i][pd["test_loss_argmin_t"][i]]) for i in am_i]
        # basis arrows: v1 is the anchor -> lands on +x as a unit vector (1, 0); v2 (the RAW
        # y-pick) is tilted off +y by its correlation (cosine) with v1. The y-AXIS itself is
        # v2 orthogonalized against v1 (v2^perp), which is why the v2 arrow leans off it.
        cos_ab = float(G[a, b] / np.sqrt(G[a, a] * G[b, b]))
        basis_arrows = {"v1": (1.0, 0.0, x_label),
                        "v2": (cos_ab, float(np.sqrt(max(1.0 - cos_ab ** 2, 0.0))), y_label)}
        return plot_trajectories_with_time(
            proj, n_ticks=n_ticks, mark_points={"w*": w_star_2d},
            w_tilde_dirs=w_tilde_dirs, early_stop_pts=es, early_stop_traj=es_i,
            argmin_pts=am, argmin_traj=am_i,
            x_label="v₁", y_label="v₂⊥", basis_arrows=basis_arrows,
            title=title or f"region run [{config_key(pd['setup'])}]  ({x_label} vs {y_label})",
        )
    # legacy v1/v2 fixed projection
    proj = pd["projected_trajectories"]
    es_i = [i for i, t in enumerate(pd["early_stop_t"]) if t is not None]
    am_i = [i for i, t in enumerate(pd["test_loss_argmin_t"]) if t is not None]
    es = [tuple(proj[i][pd["early_stop_t"][i]]) for i in es_i]
    am = [tuple(proj[i][pd["test_loss_argmin_t"][i]]) for i in am_i]
    return plot_trajectories_with_time(
        proj, n_ticks=n_ticks, mark_points={"w*": pd["w_star_proj"]},
        w_tilde_dirs=pd["w_tilde_dirs"], early_stop_pts=es, early_stop_traj=es_i,
        argmin_pts=am, argmin_traj=am_i,
        title=title or f"region run [{config_key(pd['setup'])}]",
    )


## Version 2 — with early-stopping & test-loss-argmin markers

`LLM_generated_region_experiment_v2` additionally generates a large held-out test set
(`test_sample_size`, ~population loss) and, per trajectory, records two iterates:

- **early stop** (green square) — first `t` with training loss `<= L_hat(w*_{0:k})`
  (the oracular early-stopping rule from `plot_loss_over_time`).
- **test-loss argmin** (purple star) — `argmin_t` of the population (test) logistic loss,
  i.e. the best iterate to have stopped at.

Both are indices into the trajectory (`loss[t]` lines up with `w_trajectory[t]`), so they
drop directly onto the projected path. Toggle each via its legend entry to compare where,
in the plane, the practical stop lands vs. the optimum. They usually sit deep in the
end-cluster — zoom in.


### Reading it
- Where ticks pile into a tight cluster, GD spent a huge number of steps moving very little — that\'s the slow, late-time regime.
- Drag-select a rectangle on the plot to **zoom** into that cluster; hover any marker to read its step number and watch the late steps fan apart.
- Double-click to reset the zoom.

Tune `n_ticks` (or pass `tick_every=`) to trade off clutter vs. resolution.


## Control panel

Enter a setup, pick **v1** (no stop markers) or **v2** (early-stop green square +
test-loss argmin purple star), and hit **Run / load** — it computes GD once and caches
it, or reloads instantly if that exact setup was run before. The **saved runs** dropdown
lists every cached run (selecting one refills the fields and redraws it); the `n_ticks`
slider re-renders the current run at a different tick density. `t_steps` accepts `1e5` /
`100k` / `100000`.

There is **one cache entry per setting** (everything except `t_steps`). Asking for a
larger `t_steps` on the same setting extends that single entry to the higher T; asking
for a smaller/equal `t_steps` just reloads the stored (highest-T) run.


**v3 axes.** The **v1 (x-axis)** dropdown picks the *anchor* direction — it sits exactly on the x-axis. The **v2 (y-axis)** vector is orthogonalized against v1, so it tilts off the y-axis by however much it correlates with v1 (the `v1`/`v2` arrows on the plot show the real frame). Switching axes **keeps your legend selection**, so you can isolate a few trajectories and rotate the projection to view them from different angles.

In [ ]:
def _parse_axes(s):
    """'5,10,15,50' -> sorted unique [5,10,15,50]; empty -> None (no basis axes)."""
    vals = sorted({int(x) for x in s.replace(",", " ").split()})
    return vals or None

def _parse_int(s):
    s = str(s).strip().lower()
    return int(float(s[:-1]) * 1000) if s.endswith("k") else int(float(s))

_st = {"description_width": "115px"}; _lo = widgets.Layout(width="215px")
def _T(desc, val): return widgets.Text(value=str(val), description=desc, style=_st, layout=_lo)

w_ver   = widgets.Dropdown(options=["v3", "v2", "v1"], value="v3", description="version:",
                           style=_st, layout=_lo)
w_n     = _T("n:", n);      w_d = _T("d:", d);   w_k = _T("k:", k)
w_ntraj = _T("n_traj:", number_of_trajectories); w_tstep = _T("t_steps:", t_steps)
w_eta   = _T("eta:", eta);  w_seed = _T("seed:", seed)
w_test  = _T("test_size:", int(3e3))
# number of random unit vectors added to the projection-axis set (browsable as rand_i)
w_nrand = widgets.Text(value="2", description="# rand proj axes:",
                       style={"description_width": "150px"},
                       layout=widgets.Layout(width="240px"))
# v3 only: extra selectable axes e_m (0-indexed coords), e.g. "1,2,30,50,100". Out-of-range skipped.
w_basis = widgets.Text(value="", description="basis axes e_m:",
                       style={"description_width": "150px"},
                       layout=widgets.Layout(width="260px"))
w_norm  = widgets.Checkbox(value=False, description="normalize_w_tilde", indent=False)
# v3 only: choose which stored axes become x and y. Reconstructed at view time (no GD rerun),
# so switching axes is instant. Disabled for legacy v1/v2 (they store one fixed plane).
w_xaxis = widgets.Dropdown(options=[], description="v1 (x-axis):", style=_st, layout=_lo)
w_yaxis = widgets.Dropdown(options=[], description="v2 (y-axis):", style=_st, layout=_lo)
w_ticks = widgets.IntSlider(value=400, min=50, max=1500, step=50, description="n_ticks:",
                            style=_st, layout=widgets.Layout(width="340px"))

run_btn = widgets.Button(description="Run / load", button_style="primary",
                         layout=widgets.Layout(width="120px"))
run_dd  = widgets.Dropdown(options=[], description="saved runs:", style=_st,
                           layout=widgets.Layout(width="680px"))
status  = widgets.Label(value="enter a setup and click Run / load, or pick a saved run from the dropdown.")
# persistent figure surface, updated IN PLACE (no Output widget, no re-display).
fig_widget = go.FigureWidget()
fig_widget.update_layout(width=820, height=820, template="plotly_white",
                         title="run a setup to see trajectories")

_state = {"pd": None, "key": None}   # currently displayed run

def _setup_from_inputs():
    ver = w_ver.value
    return dict(version=ver, n=_parse_int(w_n.value), d=_parse_int(w_d.value),
                k=_parse_int(w_k.value), number_of_trajectories=_parse_int(w_ntraj.value),
                t_steps=_parse_int(w_tstep.value), eta=float(w_eta.value),
                seed=_parse_int(w_seed.value),
                test_sample_size=(_parse_int(w_test.value) if ver in ("v2", "v3") else None),
                normalize_w_tilde=bool(w_norm.value), use_lambda_diag=True,
                n_random_axes=_parse_int(w_nrand.value),
                basis_axes=_parse_axes(w_basis.value))

def _fill_inputs(s):
    w_ver.value = s["version"]; w_n.value = str(s["n"]); w_d.value = str(s["d"])
    w_k.value = str(s["k"]); w_ntraj.value = str(s["number_of_trajectories"])
    w_tstep.value = str(s["t_steps"]); w_eta.value = str(s["eta"])
    w_seed.value = str(s["seed"])
    if s.get("test_sample_size") is not None: w_test.value = str(s["test_sample_size"])
    if s.get("n_random_axes") is not None: w_nrand.value = str(s["n_random_axes"])
    _ba = s.get("basis_axes"); w_basis.value = ",".join(map(str, _ba)) if _ba else ""
    w_norm.value = bool(s["normalize_w_tilde"])

def _refresh_dd():
    run_dd.unobserve(_on_select, names="value")
    idx = load_index()
    run_dd.options = [(run_label(k, s), k) for k, s in sorted(idx.items())]
    run_dd.observe(_on_select, names="value")

def _populate_axes(pd):
    labels = pd.get("axis_labels")
    w_xaxis.unobserve(_on_axis, names="value"); w_yaxis.unobserve(_on_axis, names="value")
    if labels:
        w_xaxis.options = labels; w_yaxis.options = labels
        w_xaxis.value = "w*" if "w*" in labels else labels[0]
        w_yaxis.value = "sum_w~" if "sum_w~" in labels else labels[-1]
        w_xaxis.disabled = w_yaxis.disabled = False
    else:                                    # legacy v1/v2: fixed plane, pickers off
        w_xaxis.options = []; w_yaxis.options = []
        w_xaxis.disabled = w_yaxis.disabled = True
    w_xaxis.observe(_on_axis, names="value"); w_yaxis.observe(_on_axis, names="value")

def _render(preserve=True):
    # preserve=True keeps each trace's legend on/off state (by position) across axis /
    # n_ticks changes, so you can isolate a few trajectories and rotate the view without
    # everything reappearing. New-run / new-selection calls pass preserve=False (show all).
    pd = _state["pd"]
    if pd is None: return
    xl = w_xaxis.value if w_xaxis.options else None
    yl = w_yaxis.value if w_yaxis.options else None
    new = plot_from_plotdata(pd, n_ticks=w_ticks.value, x_label=xl, y_label=yl)
    prev_vis = [t.visible for t in fig_widget.data]
    with fig_widget.batch_update():
        fig_widget.data = ()
        fig_widget.add_traces(new.data)
        fig_widget.layout = new.layout
        if preserve and len(prev_vis) == len(fig_widget.data):
            for t, v in zip(fig_widget.data, prev_vis):
                t.visible = v

def _on_run(_=None):
    status.value = "running / loading (GD can take a while the first time)..."
    try:
        pd, key = run_or_load_region(_setup_from_inputs())
    except Exception as e:
        status.value = f"error: {e}"; raise
    _state["pd"], _state["key"] = pd, key
    _refresh_dd()
    run_dd.unobserve(_on_select, names="value"); run_dd.value = key
    run_dd.observe(_on_select, names="value")
    _populate_axes(pd); _render(preserve=False)   # fresh run -> show all
    status.value = f"showing [{key}]"

def _on_select(change):
    key = change["new"]
    if key is None: return
    pd = load_region_run(key)
    _fill_inputs(pd["setup"])
    _state["pd"], _state["key"] = pd, key
    _populate_axes(pd); _render(preserve=False)   # new selection -> show all

def _on_axis(_):
    _render(preserve=True)                        # keep legend selection while rotating

def _on_ticks(_):
    _render(preserve=True)

run_btn.on_click(_on_run)
run_dd.observe(_on_select, names="value")
w_xaxis.observe(_on_axis, names="value"); w_yaxis.observe(_on_axis, names="value")
w_ticks.observe(_on_ticks, names="value")

display(widgets.VBox([
    widgets.HBox([w_ver, w_n, w_d, w_k]),
    widgets.HBox([w_ntraj, w_tstep, w_eta, w_seed]),
    widgets.HBox([w_test, w_nrand, w_basis, w_norm]),
    widgets.HBox([w_xaxis, w_yaxis, w_ticks]),
    widgets.HBox([run_btn, status]),
    run_dd,
    fig_widget,
]))
_refresh_dd()


KeyError: 'basis_axes'

# Over-time trajectory comparison (a *different* plot type)

Everything above projects trajectories onto a 2D **spatial** plane. This section is a
**time-domain** comparison instead: the x-axis is the GD step `t`.

Two GD trajectories `w¹_t`, `w²_t` are run on **independent** data draws from the same
distribution (same `w*`). We then compare them over time:

- **`angle(w¹_t, w²_t)`** — angle (radians) between the two full weight vectors.
- **`‖w¹_t − w²_t‖`** on a **coordinate subset**: over **all `d`** coords, and for each
  `k` in `k_list` the **first `k`** and the **last `d−k`** coords (so
  `‖first k‖² + ‖last d−k‖² = ‖all‖²`).

Uses the full weight vectors, **not** the `P`/`G` projection machinery. Separate cache
(`trajectory_comparison_cache/`) so it never mixes with the region runs above. All curves
start **hidden** — click legend entries to add them. Green buttons toggle linear/log on
each axis; checkboxes overlay the early-stop / test-loss-argmin vertical lines.


In [7]:
# ==== Over-time trajectory comparison — cache + run/load (separate from the region cache) ====
# A DIFFERENT plot type from the region plots above: x-axis = GD step (time), not a spatial
# projection. Two INDEPENDENT-data GD trajectories are compared over time. Its own cache dir +
# index so it never mixes with the region runs. Reuses os/json/hashlib/pickle/np already imported.
TC_CACHE_DIR = "trajectory_comparison_cache"
os.makedirs(TC_CACHE_DIR, exist_ok=True)
TC_INDEX_FILE = os.path.join(TC_CACHE_DIR, "index.json")

# A run's identity: the two trajectories depend on (n, d, signal_k, eta, seed, use_lambda_diag).
# k_list only selects coordinate splits, but we store one (T,) series per k (no full d-dim
# trajectory), so a different k_list is a different stored entry. t_steps is EXCLUDED from
# identity (more steps extend the same entry, exactly like the region cache).
TC_SETUP_KEYS = ["n", "d", "signal_k", "t_steps", "eta", "seed",
                 "test_sample_size", "use_lambda_diag", "k_list"]
TC_IDENTITY_KEYS = [key for key in TC_SETUP_KEYS if key != "t_steps"]

def tc_parse_int(s):
    s = str(s).strip().lower()
    return int(float(s[:-1]) * 1000) if s.endswith("k") else int(float(s))

def tc_parse_klist(s):
    return [int(float(x)) for x in str(s).replace(" ", "").split(",") if x != ""]

def tc_config_key(setup):
    payload = {k: setup[k] for k in TC_IDENTITY_KEYS}
    payload["k_list"] = list(payload["k_list"])          # normalize for hashing
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:8]

def tc_load_index():
    try:
        with open(TC_INDEX_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def tc_update_index(key, setup):
    idx = tc_load_index()
    entry = {k: setup[k] for k in TC_SETUP_KEYS}
    entry["k_list"] = list(entry["k_list"])
    idx[key] = entry
    with open(TC_INDEX_FILE, "w") as f:
        json.dump(idx, f, indent=2)

def tc_run_label(key, s):
    return (f"[{key}] n{s['n']} d{s['d']} sigk{s['signal_k']} T{s['t_steps']} "
            f"eta{s['eta']} seed{s['seed']} k_list={list(s['k_list'])}")

def tc_run_or_load(setup, force=False):
    """Load compact over-time series for `setup` from cache, or run the experiment + save it.
    Returns (plotdata, key). Pass force=True to recompute and overwrite the entry."""
    setup = dict(setup); setup["k_list"] = list(setup["k_list"])
    key = tc_config_key(setup)
    path = os.path.join(TC_CACHE_DIR, f"tc_{key}.pkl")
    if os.path.exists(path) and not force:
        with open(path, "rb") as f:
            pd = pickle.load(f)
        if pd["setup"]["t_steps"] >= setup["t_steps"]:   # stored entry has >= requested steps
            print("loaded", tc_run_label(key, pd["setup"]))
            return pd, key
        # asking for more steps: recompute at the larger T and overwrite the one entry.
        print(f"extending T {pd['setup']['t_steps']} -> {setup['t_steps']} (recompute)")

    w_star = np.zeros(setup["d"]); w_star[:setup["signal_k"]] = 1.0
    lam = np.arange(1, setup["d"] + 1, dtype=float) ** (-2)
    inst = LogisticRegressionClass(setup["n"], setup["d"],
                                   random_seed=setup["seed"], eta=setup["eta"])
    result = inst.LLM_generated_trajectory_comparison_over_time(
        w_star, d=setup["d"], n=setup["n"], t_steps=setup["t_steps"], eta=setup["eta"],
        use_lambda_diag=setup["use_lambda_diag"], lambda_diag=lam,
        k_list=setup["k_list"], test_sample_size=setup["test_sample_size"],
        measure_population_loss_for_iterates=True, plot=False)
    pd = {
        "setup": {k: setup[k] for k in TC_SETUP_KEYS},
        "angle": np.asarray(result["angle"]),
        "diff_all": np.asarray(result["diff_all"]),
        "diff_first_k": {int(k): np.asarray(v) for k, v in result["diff_first_k"].items()},
        "diff_last_k": {int(k): np.asarray(v) for k, v in result["diff_last_k"].items()},
        "k_list": [int(k) for k in result["k_list"]],
        "d": int(result["d"]),
        "early_stop_t": list(result["early_stop_t"]),
        "test_loss_argmin_t": list(result["test_loss_argmin_t"]),
    }
    pd["setup"]["k_list"] = list(pd["setup"]["k_list"])
    with open(path, "wb") as f:
        pickle.dump(pd, f)
    tc_update_index(key, setup)
    print("computed + saved", tc_run_label(key, pd["setup"]))
    return pd, key

def tc_load_run(key):
    with open(os.path.join(TC_CACHE_DIR, f"tc_{key}.pkl"), "rb") as f:
        return pickle.load(f)


In [8]:
# ==== Over-time trajectory comparison — control panel ====
# Everything is tc_-prefixed so it never clobbers the region control panel's globals above.
_tc_st = {"description_width": "95px"}; _tc_lo = widgets.Layout(width="200px")
def _tcT(desc, val): return widgets.Text(value=str(val), description=desc, style=_tc_st, layout=_tc_lo)

tc_w_n     = _tcT("n:", n);   tc_w_d = _tcT("d:", d)
tc_w_sigk  = _tcT("signal k:", k)                 # support of w*: first signal_k coords = 1
tc_w_tstep = _tcT("t_steps:", t_steps)
tc_w_eta   = _tcT("eta:", eta); tc_w_seed = _tcT("seed:", seed)
tc_w_test  = _tcT("test_size:", int(3e3))
# coordinate-split points; each k plots ||first k|| and ||last d-k||. Default includes signal k.
tc_w_klist = widgets.Text(value="5,10,15,50", description="k_list:",
                          style={"description_width": "95px"}, layout=widgets.Layout(width="320px"))

tc_run_btn = widgets.Button(description="Run / load", button_style="primary",
                            layout=widgets.Layout(width="120px"))
tc_run_dd  = widgets.Dropdown(options=[], description="saved runs:", style=_tc_st,
                              layout=widgets.Layout(width="680px"))
tc_status  = widgets.Label(value="enter a setup and click Run / load, or pick a saved run.")

# --- linear|log scale toggle pairs (green = selected), like logistic_visualization ---
def tc_make_toggle_pair(label, default_log=False):
    b_lin = widgets.ToggleButton(value=not default_log, description="linear",
                                 button_style=("" if default_log else "success"),
                                 layout=widgets.Layout(width="66px"))
    b_log = widgets.ToggleButton(value=default_log, description="log",
                                 button_style=("success" if default_log else ""),
                                 layout=widgets.Layout(width="56px"))
    def on_lin(change):
        if change["new"]:
            b_log.value = False; b_log.button_style = ""; b_lin.button_style = "success"
        elif not b_log.value:
            b_lin.value = True                     # keep at least one selected
        tc_update_scales()
    def on_log(change):
        if change["new"]:
            b_lin.value = False; b_lin.button_style = ""; b_log.button_style = "success"
        elif not b_lin.value:
            b_log.value = True
        tc_update_scales()
    b_lin.observe(on_lin, names="value"); b_log.observe(on_log, names="value")
    row = widgets.HBox([widgets.Label(label, layout=widgets.Layout(width="82px")), b_lin, b_log])
    return row, b_lin, b_log

tc_x_row,  tc_x_lin,  tc_x_log  = tc_make_toggle_pair("time (x):")
tc_ya_row, tc_ya_lin, tc_ya_log = tc_make_toggle_pair("angle y:")
tc_yd_row, tc_yd_lin, tc_yd_log = tc_make_toggle_pair("diff y:", default_log=True)

# stop-line toggles (default OFF). Drawn as full-height layout shapes (yref=paper) so they
# never need rescaling when the axes switch linear<->log. Green = early stop, purple = argmin.
tc_cb_es = widgets.Checkbox(value=False, description="early-stop lines", indent=False)
tc_cb_am = widgets.Checkbox(value=False, description="test-argmin lines", indent=False)

tc_fig_angle = go.FigureWidget()
tc_fig_angle.update_layout(width=840, height=330, template="plotly_white",
                           title="angle(w¹_t, w²_t) over time",
                           xaxis_title="GD step t", yaxis_title="angle [deg]", showlegend=True)
tc_fig_diff = go.FigureWidget()
tc_fig_diff.update_layout(width=840, height=380, template="plotly_white",
                          title="‖w¹_t − w²_t‖ over time (coordinate subsets)",
                          xaxis_title="GD step t", yaxis_title="norm of difference", showlegend=True)

TC_KCOLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e",
              "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]
tc_state = {"pd": None, "key": None}

def tc_update_scales():
    xt = "log" if tc_x_log.value else "linear"
    tc_fig_angle.update_layout(xaxis_type=xt, yaxis_type=("log" if tc_ya_log.value else "linear"))
    tc_fig_diff.update_layout(xaxis_type=xt,  yaxis_type=("log" if tc_yd_log.value else "linear"))

def tc_build_shapes():
    """Full-height vertical lines (yref=paper) at the early-stop / test-argmin steps of both
    trajectories. Visibility follows the checkboxes; default off."""
    pd = tc_state["pd"]
    if pd is None: return []
    shapes = []
    for t in pd["early_stop_t"]:
        if t is not None:
            shapes.append(dict(type="line", xref="x", yref="paper", x0=t, x1=t, y0=0, y1=1,
                               line=dict(color="green", dash="dash", width=1),
                               visible=tc_cb_es.value))
    for t in pd["test_loss_argmin_t"]:
        if t is not None:
            shapes.append(dict(type="line", xref="x", yref="paper", x0=t, x1=t, y0=0, y1=1,
                               line=dict(color="purple", dash="dot", width=1),
                               visible=tc_cb_am.value))
    return shapes

def tc_render():
    pd = tc_state["pd"]
    if pd is None: return
    ts = list(range(len(pd["angle"])))
    with tc_fig_angle.batch_update():
        tc_fig_angle.data = ()
        tc_fig_angle.add_trace(go.Scatter(x=ts, y=list(np.degrees(pd["angle"])), mode="lines",
                                          name="angle(w¹,w²)", visible="legendonly",
                                          line=dict(color="black")))
        tc_fig_angle.layout.shapes = tc_build_shapes()
    with tc_fig_diff.batch_update():
        tc_fig_diff.data = ()
        tc_fig_diff.add_trace(go.Scatter(x=ts, y=list(pd["diff_all"]), mode="lines",
                                         name="all d", visible="legendonly",
                                         line=dict(color="black", width=2)))
        for j, kk in enumerate(pd["k_list"]):
            c = TC_KCOLORS[j % len(TC_KCOLORS)]
            tc_fig_diff.add_trace(go.Scatter(x=ts, y=list(pd["diff_first_k"][kk]), mode="lines",
                                             name=f"first {kk}", visible="legendonly",
                                             line=dict(color=c)))
            tc_fig_diff.add_trace(go.Scatter(x=ts, y=list(pd["diff_last_k"][kk]), mode="lines",
                                             name=f"last d-{kk}", visible="legendonly",
                                             line=dict(color=c, dash="dash")))
        tc_fig_diff.layout.shapes = tc_build_shapes()
    tc_update_scales()

def tc_on_stoplines(_=None):
    with tc_fig_angle.batch_update():
        tc_fig_angle.layout.shapes = tc_build_shapes()
    with tc_fig_diff.batch_update():
        tc_fig_diff.layout.shapes = tc_build_shapes()

def tc_setup_from_inputs():
    return dict(n=tc_parse_int(tc_w_n.value), d=tc_parse_int(tc_w_d.value),
                signal_k=tc_parse_int(tc_w_sigk.value), t_steps=tc_parse_int(tc_w_tstep.value),
                eta=float(tc_w_eta.value), seed=tc_parse_int(tc_w_seed.value),
                test_sample_size=tc_parse_int(tc_w_test.value),
                use_lambda_diag=True, k_list=tc_parse_klist(tc_w_klist.value))

def tc_fill_inputs(s):
    tc_w_n.value = str(s["n"]); tc_w_d.value = str(s["d"]); tc_w_sigk.value = str(s["signal_k"])
    tc_w_tstep.value = str(s["t_steps"]); tc_w_eta.value = str(s["eta"]); tc_w_seed.value = str(s["seed"])
    tc_w_test.value = str(s["test_sample_size"])
    tc_w_klist.value = ",".join(str(x) for x in s["k_list"])

def tc_refresh_dd():
    tc_run_dd.unobserve(tc_on_select, names="value")
    idx = tc_load_index()
    tc_run_dd.options = [(tc_run_label(k_, s), k_) for k_, s in sorted(idx.items())]
    tc_run_dd.observe(tc_on_select, names="value")

def tc_on_run(_=None):
    tc_status.value = "running / loading (GD can take a while the first time)..."
    try:
        pd, key = tc_run_or_load(tc_setup_from_inputs())
    except Exception as e:
        tc_status.value = f"error: {e}"; raise
    tc_state["pd"], tc_state["key"] = pd, key
    tc_refresh_dd()
    tc_run_dd.unobserve(tc_on_select, names="value"); tc_run_dd.value = key
    tc_run_dd.observe(tc_on_select, names="value")
    tc_render()
    tc_status.value = f"showing [{key}] — all curves start hidden; click legend entries to add them"

def tc_on_select(change):
    key = change["new"]
    if key is None: return
    pd = tc_load_run(key)
    tc_fill_inputs(pd["setup"])
    tc_state["pd"], tc_state["key"] = pd, key
    tc_render()

tc_run_btn.on_click(tc_on_run)
tc_run_dd.observe(tc_on_select, names="value")
tc_cb_es.observe(tc_on_stoplines, names="value")
tc_cb_am.observe(tc_on_stoplines, names="value")

display(widgets.VBox([
    widgets.HBox([tc_w_n, tc_w_d, tc_w_sigk, tc_w_seed]),
    widgets.HBox([tc_w_tstep, tc_w_eta, tc_w_test]),
    widgets.HBox([tc_w_klist]),
    widgets.HBox([tc_x_row, tc_ya_row, tc_yd_row]),
    widgets.HBox([tc_cb_es, tc_cb_am]),
    widgets.HBox([tc_run_btn, tc_status]),
    tc_run_dd,
    tc_fig_angle,
    tc_fig_diff,
]))
tc_refresh_dd()
